# Fetch NWM v3.1 streamflow for the post-cutover gap (#439)

The Prefect ingests kept running as `nwm30` until #396 (2026-09-22), so the warehouse has no
`nwm31_*` streamflow from the v3.1 cutover (2026-08-18 t00z) until the `nwm31` ingests took
over. This notebook fetches that gap to parquet. `23_load_backfill_nwm31_streamflow.ipynb`
then loads it, and `04_maintenance/11_delete_nwm30_after_v31_cutover.ipynb` removes the
mislabeled `nwm30_*` rows.

Streamflow (point) configurations only.

Run it on the 4xlarge server.

In [ ]:
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.const import (
    NWM_HAWAII_VARIABLE_MAPPER,
    NWM_VARIABLE_MAPPER,
    VARIABLE_NAME,
)
from teehr.fetching.nwm.nwm_points import nwm_to_parquet
from teehr.fetching.utils import format_nwm_configuration_metadata

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

teehr.__version__

In [ ]:
NWM_VERSION = "nwm31"
CUTOVER = datetime(2026, 8, 18, 0)  # teehr NWM31_START_DATE

# Persistent storage, so built kerchunk references and written chunks survive a restart.
BACKFILL_DIR = Path("/data/data_processing/backfill-nwm31")
KERCHUNK_DIR = BACKFILL_DIR / "kerchunk"
PARQUET_DIR = BACKFILL_DIR / "parquet"

OCONUS_STATE_NAMES = [
    'Northern Mariana Islands', 'Alaska', 'Hawaii', 'Guam',
    'American Samoa', 'Puerto Rico', 'Virgin Islands'
]
DOMAIN_STATE_NAMES = {
    "hawaii": "Hawaii",
    "alaska": "Alaska",
    "puertorico": "Puerto Rico",
}

CONFIGS = [
    ("short_range", "channel_rt"),
    ("short_range_alaska", "channel_rt"),
    ("short_range_hawaii", "channel_rt"),
    ("short_range_puertorico", "channel_rt"),
    ("medium_range_mem1", "channel_rt_1"),
    ("medium_range_blend", "channel_rt"),
    ("medium_range_alaska_mem1", "channel_rt_1"),
    ("medium_range_blend_alaska", "channel_rt"),
    ("analysis_assim_no_da", "channel_rt"),
    ("analysis_assim_alaska_no_da", "channel_rt"),
    ("analysis_assim_extend_no_da", "channel_rt"),
    ("analysis_assim_extend_alaska_no_da", "channel_rt"),
    ("analysis_assim_hawaii_no_da", "channel_rt"),
    ("analysis_assim_puertorico_no_da", "channel_rt"),
]


def config_domain(nwm_configuration: str) -> str:
    """Domain of a configuration, matched the same way as the ingest flow."""
    for domain in DOMAIN_STATE_NAMES:
        if domain in nwm_configuration:
            return domain
    return "conus"


def ids_csv(domain: str) -> Path:
    return BACKFILL_DIR / f"{NWM_VERSION}_{domain}_ids.csv"

#### One time: get the NWM v3.1 location IDs for each domain and save to csv
Same crosswalk filter as the ingest flow's `_filter_crosswalk_table`. The IDs have to come
from the `nwm31` crosswalk, because v3.1 feature IDs differ from v3.0.

In [ ]:
%%time
spark = create_spark_session()  # For read-only
ev = teehr.RemoteReadOnlyEvaluation(spark=spark)

In [ ]:
%%time
BACKFILL_DIR.mkdir(parents=True, exist_ok=True)
oconus_states = ", ".join(f"'{s}'" for s in OCONUS_STATE_NAMES)
domain_filters = {
    "conus": f"state_name NOT IN ({oconus_states})",
    **{d: f"state_name = '{s}'" for d, s in DOMAIN_STATE_NAMES.items()},
}
for domain, state_filter in domain_filters.items():
    filtered_crosswalks_sdf = ev.location_crosswalks.add_attributes(
        attr_list=["state_name"]
    ).filter(
        filters=[
            {
                "column": "secondary_location_id",
                "operator": "like",
                "value": f"{NWM_VERSION}-%"
            },
            state_filter
        ]
    ).to_sdf()
    stripped_ids = [
        row[0].split("-")[1]
        for row in filtered_crosswalks_sdf.select("secondary_location_id").collect()
    ]
    pd.DataFrame({"nwm_id": stripped_ids}).to_csv(ids_csv(domain), index=False)
    print(f"{domain}: {len(stripped_ids)} IDs")

#### Find the end of the gap
The earliest `nwm31_*` data from the scheduled ingest is where each backfill ends. The
`nwm30_*` rows after the cutover are what the delete notebook will remove. Analysis configs
have a NULL `reference_time` (overlapping values are dropped), so `value_time` is shown too.

In [ ]:
%%time
config_names = []
for nwm_configuration, _ in CONFIGS:
    for version in ("nwm30", NWM_VERSION):
        config_names.append(
            format_nwm_configuration_metadata(nwm_configuration, version)["name"]
        )
names_sql = ", ".join(f"'{n}'" for n in config_names)
coverage_df = ev.spark.sql(f"""
    SELECT
        configuration_name,
        MIN(reference_time) AS min_reference_time,
        MAX(reference_time) AS max_reference_time,
        MIN(value_time) AS min_value_time,
        MAX(value_time) AS max_value_time,
        COUNT_IF(reference_time >= '{CUTOVER}') AS n_ref_after_cutover,
        COUNT_IF(reference_time IS NULL AND value_time >= '{CUTOVER}') AS n_null_ref_after_cutover
    FROM iceberg.teehr.secondary_timeseries
    WHERE configuration_name IN ({names_sql})
        AND value_time >= '{CUTOVER}'
    GROUP BY configuration_name
    ORDER BY configuration_name
""").toPandas()
coverage_df

In [ ]:
ev.spark.stop()

#### Fetch one configuration
Set the configuration and the end of its gap from the table above. `END_DATE` is the last
cycle to fetch, inclusive. Setting it to the first `nwm31` cycle already in the warehouse is
fine: the load drops the duplicates.

`process_by_z_hour=False` groups `STEPSIZE` files per chunk rather than one cycle per chunk.
Chunks are read one at a time, so a one-cycle chunk (18 files for short_range, fewer for
analysis) leaves most of the read budget idle. A chunk can then span several cycles, which
the load does not care about.

Kerchunk references missing from the CIROH bucket are built locally first (roughly 3-5
files/s) and cached in `KERCHUNK_DIR`. Re-running with the same dates and `STEPSIZE` skips
the chunks already written, so an interrupted run resumes where it stopped.

In [ ]:
NWM_CONFIGURATION = "short_range"
START_DATE = CUTOVER
END_DATE = datetime(2026, 9, 18, 23)  # From the coverage table

VARIABLE_NAME_NWM = "streamflow"
STEPSIZE = 500
# Remote reads in flight. None uses teehr's default (48).
IO_CONCURRENCY = None

In [ ]:
output_type = dict(CONFIGS)[NWM_CONFIGURATION]
domain = config_domain(NWM_CONFIGURATION)
location_ids = pd.read_csv(ids_csv(domain))["nwm_id"].tolist()

if domain == "hawaii":
    variable_mapper = NWM_HAWAII_VARIABLE_MAPPER
else:
    variable_mapper = NWM_VARIABLE_MAPPER
ev_variable_name = variable_mapper[VARIABLE_NAME].get(
    VARIABLE_NAME_NWM, {}
).get("name", VARIABLE_NAME_NWM)
ev_config = format_nwm_configuration_metadata(
    nwm_config_name=NWM_CONFIGURATION,
    nwm_version=NWM_VERSION
)
output_parquet_dir = Path(PARQUET_DIR, ev_config["name"], ev_variable_name)

print(f"{NWM_CONFIGURATION} ({output_type}), {domain}: {len(location_ids)} locations")
print(f"{START_DATE} to {END_DATE} -> {output_parquet_dir}")

In [ ]:
%%time
nwm_to_parquet(
    configuration=NWM_CONFIGURATION,
    output_type=output_type,
    variable_name=VARIABLE_NAME_NWM,
    start_date=START_DATE,
    end_date=END_DATE,
    location_ids=location_ids,
    json_dir=KERCHUNK_DIR,
    output_parquet_dir=output_parquet_dir,
    nwm_version=NWM_VERSION,
    variable_mapper=variable_mapper,
    kerchunk_method="auto",
    process_by_z_hour=False,
    stepsize=STEPSIZE,
    overwrite_output=False,
    drop_overlapping_assimilation_values=True,
    io_concurrency=IO_CONCURRENCY,
)

#### Check the output
Reads single columns per file, so this stays cheap on the biggest configurations.

In [ ]:
%%time
files = sorted(output_parquet_dir.glob("*.parquet"))
n_rows = 0
ref_times, value_times = set(), []
for f in files:
    tbl = pq.read_table(f, columns=["reference_time", "value_time"])
    n_rows += tbl.num_rows
    ref_times.update(tbl.column("reference_time").unique().to_pylist())
    vt = tbl.column("value_time")
    value_times += [vt[0].as_py(), vt[-1].as_py()]
sample = pq.read_table(files[0]).to_pandas()

print(f"Files: {len(files)}, {sum(f.stat().st_size for f in files) / 1024**3:.2f} GB, {n_rows:,} rows")
print(f"Distinct reference_time: {len(ref_times)}")
print(f"value_time: {min(value_times)} to {max(value_times)}")
print(f"Locations in first file: {sample.location_id.nunique()} of {len(location_ids)} requested")
sample.head()